In [32]:
import os
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [33]:
SEED = 42
BATCH_SIZE = 2048
EPOCHS = 20

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

device = torch.device("cuda")
print("Device:", device)

Device: cuda


In [34]:
PROJECT_ROOT = Path(".")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
FIGURES_DIR = ARTIFACTS_DIR / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [35]:
transform = transforms.ToTensor()

DATA_ROOT = "../HW08/data"

transform = transforms.ToTensor()

train_full = torchvision.datasets.EMNIST(
    root=DATA_ROOT,
    split="balanced",
    train=True,
    download=False,
    transform=transform
)

test_dataset = torchvision.datasets.EMNIST(
    root=DATA_ROOT,
    split="balanced",
    train=False,
    download=False,
    transform=transform
)

num_classes = 47
input_dim = 28 * 28

In [36]:
generator = torch.Generator().manual_seed(SEED)

indices = torch.randperm(len(train_full), generator=generator)

x_data = torch.stack([train_full[i][0] for i in indices])
y_data = torch.tensor([train_full[i][1] for i in indices])

train_size = int(0.8 * len(train_full))

x_train = x_data[:train_size]
y_train = y_data[:train_size]

x_val = x_data[train_size:]
y_val = y_data[train_size:]

x_train = x_train.to(device)
y_train = y_train.to(device)

x_val = x_val.to(device)
y_val = y_val.to(device)

print("Train:", x_train.shape)
print("Val:", x_val.shape)

Train: torch.Size([90240, 1, 28, 28])
Val: torch.Size([22560, 1, 28, 28])


In [37]:
x_test = torch.stack([x for x, _ in test_dataset]).to(device)
y_test = torch.tensor([y for _, y in test_dataset]).to(device)

In [38]:
class MLP(nn.Module):
    def __init__(self, hidden_sizes, dropout=0.0, batchnorm=False):
        super().__init__()

        layers = []
        prev = input_dim

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if batchnorm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, num_classes))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        x = torch.flatten(x, 1)
        return self.net(x)

In [39]:
def train_epoch(model, optimizer, criterion):
    model.train()
    perm = torch.randperm(x_train.size(0), device=device)

    total_loss = 0
    correct = 0

    for i in range(0, x_train.size(0), BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        xb = x_train[idx]
        yb = y_train[idx]

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()

    return total_loss / x_train.size(0), correct / x_train.size(0)


@torch.no_grad()
def evaluate(model):
    model.eval()

    total_loss = 0
    correct = 0
    criterion = nn.CrossEntropyLoss()

    for i in range(0, x_val.size(0), BATCH_SIZE):
        xb = x_val[i:i+BATCH_SIZE]
        yb = y_val[i:i+BATCH_SIZE]

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()

    return total_loss / x_val.size(0), correct / x_val.size(0)

In [40]:
def run_experiment(exp_id, hidden, dropout, bn,
                   optimizer_name, lr,
                   momentum=0.0, wd=0.0,
                   epochs=EPOCHS,
                   early_stop=False,
                   save=False):

    print("=" * 70)
    print(f"Starting experiment {exp_id}")
    print("=" * 70)

    model = MLP(hidden, dropout, bn).to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr,
                              momentum=momentum, weight_decay=wd)

    history = {"train_loss": [], "val_loss": [],
               "train_acc": [], "val_acc": []}

    best_acc = 0
    best_loss = float("inf")
    best_state = None
    patience = 5
    counter = 0

    import time
    torch.cuda.synchronize()
    start_time = time.time()

    for epoch in range(epochs):

        train_loss, train_acc = train_epoch(model, optimizer, criterion)
        val_loss, val_acc = evaluate(model)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"[{exp_id}] Epoch {epoch+1}/{epochs} | "
              f"Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            best_loss = val_loss
            best_state = model.state_dict()
            counter = 0
        else:
            counter += 1

        if early_stop and counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    torch.cuda.synchronize()
    total_time = time.time() - start_time

    epochs_trained = len(history["train_loss"])

    print(f"Finished {exp_id} | Best Val Acc: {best_acc:.4f} | "
          f"Time: {total_time:.2f}s")

    if save:
        torch.save(best_state, ARTIFACTS_DIR / "best_model.pt")

        config = {
            "dataset": "EMNIST-balanced",
            "seed": SEED,
            "hidden_sizes": hidden,
            "dropout": dropout,
            "batchnorm": bn,
            "optimizer": optimizer_name,
            "lr": lr,
            "momentum": momentum,
            "weight_decay": wd,
            "epochs_trained": epochs_trained,
            "best_val_accuracy": float(best_acc)
        }

        with open(ARTIFACTS_DIR / "best_config.json", "w") as f:
            json.dump(config, f, indent=4)

    return history, best_acc, best_loss, epochs_trained, total_time

In [ ]:
hidden = [1024, 512, 256]
runs = []

E1 = run_experiment("E1", hidden, 0.0, False, "Adam", 1e-3)
E2 = run_experiment("E2", hidden, 0.3, False, "Adam", 1e-3)
E3 = run_experiment("E3", hidden, 0.0, True,  "Adam", 1e-3)

if E2[1] >= E3[1]:
    best_dropout = 0.3
    best_bn = False
else:
    best_dropout = 0.0
    best_bn = True

E4 = run_experiment("E4", hidden, best_dropout, best_bn,
                    "Adam", 1e-3,
                    epochs=50,
                    early_stop=True,
                    save=True)

O1 = run_experiment("O1", hidden, best_dropout, best_bn,
                    "Adam", 1e-1, epochs=8)

O2 = run_experiment("O2", hidden, best_dropout, best_bn,
                    "Adam", 1e-5, epochs=8)

O3 = run_experiment("O3", hidden, best_dropout, best_bn,
                    "SGD", 1e-2, momentum=0.9,
                    wd=1e-4, epochs=15)

Starting experiment E1
[E1] Epoch 1/20 | Train Acc: 0.4470 | Val Acc: 0.6146
[E1] Epoch 2/20 | Train Acc: 0.6644 | Val Acc: 0.6826
[E1] Epoch 3/20 | Train Acc: 0.7329 | Val Acc: 0.7498
[E1] Epoch 4/20 | Train Acc: 0.7736 | Val Acc: 0.7762
[E1] Epoch 5/20 | Train Acc: 0.7977 | Val Acc: 0.8013
[E1] Epoch 6/20 | Train Acc: 0.8161 | Val Acc: 0.8034
[E1] Epoch 7/20 | Train Acc: 0.8273 | Val Acc: 0.8169
[E1] Epoch 8/20 | Train Acc: 0.8387 | Val Acc: 0.8209
[E1] Epoch 9/20 | Train Acc: 0.8479 | Val Acc: 0.8161
[E1] Epoch 10/20 | Train Acc: 0.8494 | Val Acc: 0.8301
[E1] Epoch 11/20 | Train Acc: 0.8582 | Val Acc: 0.8293
[E1] Epoch 12/20 | Train Acc: 0.8634 | Val Acc: 0.8293
[E1] Epoch 13/20 | Train Acc: 0.8666 | Val Acc: 0.8426
[E1] Epoch 14/20 | Train Acc: 0.8727 | Val Acc: 0.8425
[E1] Epoch 15/20 | Train Acc: 0.8793 | Val Acc: 0.8371
[E1] Epoch 16/20 | Train Acc: 0.8834 | Val Acc: 0.8402
[E1] Epoch 17/20 | Train Acc: 0.8839 | Val Acc: 0.8508
[E1] Epoch 18/20 | Train Acc: 0.8896 | Val Acc: 0.8

In [42]:
def add_run(exp_id, result, optimizer, lr, momentum, wd):
    history, best_acc, best_loss, epochs_trained, total_time = result

    runs.append({
        "experiment_id": exp_id,
        "dataset": "EMNIST-balanced",
        "seed": SEED,
        "model_summary": f"{hidden}, dropout={best_dropout}, bn={best_bn}",
        "optimizer": optimizer,
        "lr": lr,
        "momentum": momentum,
        "weight_decay": wd,
        "epochs_trained": epochs_trained,
        "best_val_accuracy": best_acc,
        "best_val_loss": best_loss,
        "total_training_time_sec": total_time
    })

add_run("E1", E1, "Adam", 1e-3, 0, 0)
add_run("E2", E2, "Adam", 1e-3, 0, 0)
add_run("E3", E3, "Adam", 1e-3, 0, 0)
add_run("E4", E4, "Adam", 1e-3, 0, 0)
add_run("O1", O1, "Adam", 1e-1, 0, 0)
add_run("O2", O2, "Adam", 1e-5, 0, 0)
add_run("O3", O3, "SGD", 1e-2, 0.9, 1e-4)

df = pd.DataFrame(runs)
df.to_csv(ARTIFACTS_DIR / "runs.csv", index=False)

In [43]:
history_E4 = E4[0]

plt.figure()
plt.plot(history_E4["train_loss"], label="train_loss")
plt.plot(history_E4["val_loss"], label="val_loss")
plt.legend()
plt.title("Best model (E4)")
plt.savefig(FIGURES_DIR / "curves_best.png")
plt.close()

In [44]:
plt.figure()
plt.plot(O1[0]["train_loss"], label="O1 - large LR")
plt.plot(O2[0]["train_loss"], label="O2 - small LR")
plt.legend()
plt.title("LR extremes")
plt.savefig(FIGURES_DIR / "curves_lr_extremes.png")
plt.close()

In [45]:
best_model = MLP(hidden, best_dropout, best_bn).to(device)
best_model.load_state_dict(torch.load(ARTIFACTS_DIR / "best_model.pt"))

@torch.no_grad()
def test_model():
    correct = 0
    for i in range(0, x_test.size(0), BATCH_SIZE):
        xb = x_test[i:i+BATCH_SIZE]
        yb = y_test[i:i+BATCH_SIZE]
        logits = best_model(xb)
        correct += (logits.argmax(1) == yb).sum().item()
    return correct / x_test.size(0)

test_accuracy = test_model()
print("Final TEST accuracy:", test_accuracy)

Final TEST accuracy: 0.8371276595744681
